In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT=Path.cwd().parents[1]
sys.path.insert(0,str(ROOT))

import common as cfg
import helper_funcs as hf

CONFIG=cfg.load_config()
INPUT,OUTPUT,SRC=cfg.task_paths()
OUTPUT.mkdir(parents=True,exist_ok=True)
rng=cfg.initialize(CONFIG['seed'])


In [ ]:
targets=pd.read_csv(INPUT/'b_prepare'/'empirical_targets.csv')
followup=pd.read_csv(INPUT/'b_prepare'/'followup_distribution.csv')
particles=pd.read_csv(INPUT/'h_abc_smc'/'abc_particles.csv')


In [ ]:
draws,metrics=hf.posterior_predictive(particles,CONFIG,followup,targets,rng)
check=hf.posterior_predictive_check(metrics,targets)
screening,by_block=hf.model_screening(check)


In [ ]:
draws.to_csv(OUTPUT/'posterior_predictive_draws.csv',index=False)
metrics.to_csv(OUTPUT/'posterior_predictive_metrics.csv',index=False)
check.to_csv(OUTPUT/'posterior_predictive_check.csv',index=False)
screening.to_csv(OUTPUT/'promising_hypotheses.csv',index=False)
by_block.to_csv(OUTPUT/'hypothesis_block_screening.csv',index=False)


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,5))
sns.barplot(data=screening,y='model',x='median_block_error',ax=axes[0])
sns.scatterplot(data=by_block,x='block_error',y='block',hue='model',style='model',s=70,ax=axes[1])
axes[0].set(xlabel='median posterior-predictive block error',ylabel='')
axes[1].set(xlabel='median standardized error',ylabel='')
fig.tight_layout()
fig.savefig(OUTPUT/'posterior_predictive_screening.png',dpi=220)
plt.close(fig)
screening
